# B2-Li 760 r8: четыре опоры нетронутой области

Эксперимент `disentangle_b2_li760_r8_reference_ft`: небольшая корректирующая голова
на JPEG-признаках stride 8, до RGB fusion. Четыре обучаемые карты выбора опор,
предсказанная вероятность нетронутой области, cosine similarity и коррекция logits.
GT участвует только в loss; inference использует тот же предсказанный выбор опор.
Начальная коррекция нулевая, общие веса загружаются строго из EMA исходного long-рана.

6 × 24 000 примеров. LR encoder 1e-5, JPEG/decoder 3e-5, новой головы 3e-4.
Обычный sampling, 25% негативов, full-frame. JPEG-аугментации исходного long-рана:
30% Q60–99, без сдвига сетки. Отдельный JPEG-дотюн сюда не подмешивается.
Reference loss: 0.1 × (BCE нетронутости + штраф за загрязнение опор + 0.1 × overlap опор).
PNG исключаются из reference loss и коррекции. Если нет чистой ячейки GT,
штраф за загрязнение и overlap отключаются; BCE продолжает обучать отсутствие опоры.

Для контроля выберите в следующей ячейке
`experiments/disentangle_b2_li760_r8_reference_control_ft`: те же данные, аугментации,
LR старых модулей и число updates, но без новой головы. Сравнивать и с контролем, и с исходным long.
Инициализация новой головы сохраняет CPU RNG, используемый sampler/loader, относительно контроля.

Проверять AIC/Dice/FPR, plain/nonunit, остальные домены и unit, исправления/регрессии,
а также reference_authenticity/reference_contamination/reference_diversity в train loss.
Лимит FLOPs проверяется на native JPEG Full HD; это не измерение задержки на H100.
Последняя ячейка запускает обучение. Повторный запуск продолжает собственный last.pt.


In [ ]:
import numpy as np  # Import before torch on Windows (MKL initialization).
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Open this notebook from the project root or notebooks directory')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)

from src.config import load_experiment_config

experiment = 'experiments/disentangle_b2_li760_r8_reference_ft'
cfg = load_experiment_config(ROOT / 'configs' / f'{experiment}.yaml')
print('Run:', cfg.run_name)
print('Encoder / RGB:', cfg.model.encoder, cfg.dataset.image_size)
print('Epochs / full passes:', cfg.train.epochs, cfg.train.full_pass_epochs)
print('Devices / batch per GPU / accumulation:', cfg.train.devices, cfg.train.batch_size, cfg.train.grad_accum_steps)
print('Data:', cfg.paths.data_path)
print('Runs:', cfg.paths.runs_path)
cfg


In [ ]:
import torch
from src.training.builders import build_model
from src.budget import count_gflops
from src.eval.protocol import EvaluationProtocol

protocol = EvaluationProtocol.load(cfg.dataset.protocol_path)
print('Train/development:', len(protocol.rows('train')), len(protocol.rows('development')))
native_size = (1080, 1920)
with torch.device('meta'):
    budget_model = build_model(cfg.model, aux_weight=cfg.loss.aux_weight, pretrained=False).eval()
    gflops = count_gflops(budget_model, cfg.dataset.image_size,
                          native_size=native_size)
del budget_model
assert gflops <= 100, f'{gflops:.2f} GFLOPs exceeds 100'
print(f'Full inference: {gflops:.3f} GFLOPs')
if native_size is not None:
    print('Reference native JPEG size:', native_size, '; larger sources may exceed 100 GFLOPs')


In [ ]:
checkpoint = cfg.paths.runs_path / cfg.train.finetune_from
assert checkpoint.is_file(), f'Missing source checkpoint: {checkpoint}'
last = cfg.paths.runs_path / cfg.run_name / 'ckpt' / 'last.pt'
print('Resume:' if cfg.train.resume and last.is_file() else 'Initialize from:',
      last if cfg.train.resume and last.is_file() else checkpoint)
print('Reference head:', cfg.model.pristine_reference, '; loss weight:', cfg.loss.reference_weight)
print('LR encoder / JPEG / decoder / reference:', cfg.train.encoder_lr, cfg.train.jpeg_lr,
      cfg.train.head_lr, cfg.train.reference_lr)
print('Augmentations:', cfg.augmentation)
print('Negative fraction:', cfg.train.negative_fraction)


In [ ]:
from src.training.engine import run_experiment

run = run_experiment(cfg)
run.summary
